# Barrido en longitud de onda — SmartLight
Ejecutar celda a celda. El SmartLight corre `medir_loop.py` sin cambios.
Las potencias se leen con los fotodetectores internos del SmartLight.

## 0. Configuración

In [ ]:
# ── Láser (GPIB) ────────────────────────────────────────────────────────
LASER_RESOURCE  = 'GPIB1::1::INSTR'
INPUT_POWER_DBM = -3
LASER_WAIT_S    = 3.0         # segundos de espera tras cambiar lambda

# ── Barrido en longitud de onda ─────────────────────────────────────────
LAMBDA_START_NM = 1549.9
LAMBDA_STOP_NM  = 1550.1
N_PUNTOS        = 21

# ── SmartLight (SSH) ────────────────────────────────────────────────────
SL_HOST      = '10.42.0.120'
SL_USER      = 'smartlight'
SL_PASSWORD  = 'smartlight'
SL_KEY_FILE  = None
SL_WORK_DIR  = '/home/smartlight/Oxel/VIGO'

# ── Medición ────────────────────────────────────────────────────────────
INPORT      = 0
OUTPORT_POL = 1    # puerto de prueba para ajuste de polarización

# ── Parámetros del barrido ──────────────────────────────────────────────
N_RETOS  = 25    # cuántos retos distintos se miden
N_SWEEPS = 10    # cuántas veces se repite el barrido completo por cada reto

# ── Salida ──────────────────────────────────────────────────────────────
from pathlib import Path
OUT_DIR  = Path('.')
CHIP_ID  = 'Valencia_219_sl_1'

print('Configuración cargada.')
print(f'Total medidas: {N_RETOS} retos x {N_SWEEPS} sweeps x {N_PUNTOS} lambdas = '
      f'{N_RETOS * N_SWEEPS * N_PUNTOS}')

## 1. Imports

In [ ]:
import json, time, sys
import numpy as np
import matplotlib.pyplot as plt
import pyvisa
import paramiko
from datetime import datetime

print('Imports OK.')

## 2. Conectar al láser

In [ ]:
rm    = pyvisa.ResourceManager()
laser = rm.open_resource(LASER_RESOURCE)

laser.write('DBM')
laser.write('ENABLE')
laser.write(f'P={INPUT_POWER_DBM}')
laser.write(f'L={LAMBDA_START_NM}')

print(f'Laser conectado: {LASER_RESOURCE}')
print(f'P={INPUT_POWER_DBM} dBm,  lambda={LAMBDA_START_NM} nm')

## 3. Conectar al SmartLight por SSH y lanzar medir_loop.py

Esperar hasta que el chip termine de calibrar (30-60 s).

In [ ]:
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

if SL_KEY_FILE:
    ssh.connect(SL_HOST, username=SL_USER, key_filename=SL_KEY_FILE)
else:
    ssh.connect(SL_HOST, username=SL_USER, password=SL_PASSWORD)

print(f'SSH conectado a {SL_HOST}') 

chan = ssh.get_transport().open_session()
chan.get_pty()
chan.exec_command(f'cd {SL_WORK_DIR} && python3 medir_loop_po_est.py')

buf_stderr = b''
print('Esperando inicializacion del chip...')
while True:
    if chan.recv_stderr_ready():
        chunk = chan.recv_stderr(4096)
        buf_stderr += chunk
        print(chunk.decode(errors='replace'), end='')
        if b'Listo. Esperando comandos' in buf_stderr:
            print('\nChip listo.')
            break
    time.sleep(0.1)

### Función auxiliar

In [ ]:
def sl_send(cmd: dict) -> dict:
    chan.sendall((json.dumps(cmd) + '\n').encode())
    buf = b''
    while b'\n' not in buf:
        if chan.recv_ready():
            buf += chan.recv(4096)
        elif chan.recv_stderr_ready():
            msg = chan.recv_stderr(4096)
            print('[SL]', msg.decode(errors='replace'), end='')
        else:
            time.sleep(0.05)
    return json.loads(buf.split(b'\n')[0])

print('sl_send() lista.')

## 4. Ajuste de polarización

### 4a. preparar_externo — ajuste manual con medidor analógico

In [ ]:
resp = sl_send({'accion': 'preparar_externo', 'inport': INPORT, 'outport': OUTPORT_POL})
print(resp['status'])
print(f'Ajusta la polarizacion en el puerto {OUTPORT_POL} y ejecuta la celda siguiente.')

### 4b. polarizacion — verificar con fotodetectores internos

Repite cuantas veces necesites. Cuando estés satisfecho, pasa a la celda 5.

In [ ]:
resp_pol   = sl_send({'accion': 'polarizacion', 'inport': INPORT, 'outport': OUTPORT_POL})
powers_pol = {int(k): v for k, v in resp_pol['powers'].items()}
sorted_pts = sorted(powers_pol.items(), key=lambda x: x[1], reverse=True)

print(f'Potencias desde inport={INPORT} (top 10):')
for port, pwr in sorted_pts[:10]:
    marca = ' <-- OUTPORT_POL' if port == OUTPORT_POL else ''
    print(f'  puerto {port:2d}: {pwr:7.2f} dBm{marca}')

all_ports = [p for p, _ in sorted_pts]
all_pwrs  = [v for _, v in sorted_pts]
colors    = ['tomato' if p == OUTPORT_POL else 'steelblue' for p in all_ports]

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(range(len(all_ports)), all_pwrs, color=colors)
ax.set_xticks(range(len(all_ports)))
ax.set_xticklabels(all_ports, fontsize=7)
ax.set_xlabel('Puerto de salida')
ax.set_ylabel('Potencia (dBm)')
ax.set_title(f'Polarizacion: inport={INPORT}, puerto prueba={OUTPORT_POL} (rojo)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Barrido en longitud de onda

Para cada uno de los `N_RETOS` retos, fija el reto en el chip y repite el barrido completo
`N_SWEEPS` veces. Sirve para caracterizar la huella espectral de cada reto y comprobar
su estabilidad (drift, ruido, temperatura) a lo largo de los 50 barridos.

Guarda a disco después de cada barrido completo.

In [ ]:
lambdas = np.linspace(LAMBDA_START_NM, LAMBDA_STOP_NM, N_PUNTOS)

OUT_DIR.mkdir(parents=True, exist_ok=True)
ts    = datetime.now().strftime('%Y%m%d_%H%M%S')
fname = OUT_DIR / f'barrido_lambda_port{INPORT:02d}_{ts}.json'

resultados = {
    'chip_id':         CHIP_ID,
    'timestamp':       ts,
    'inport':          INPORT,
    'n_retos':         N_RETOS,
    'n_sweeps':        N_SWEEPS,
    'lambda_start':    LAMBDA_START_NM,
    'lambda_stop':     LAMBDA_STOP_NM,
    'n_puntos':        N_PUNTOS,
    'input_power_dbm': INPUT_POWER_DBM,
    # Lista de N_RETOS entradas; cada una contiene N_SWEEPS barridos
    'retos': [],
}

t0 = time.time()

for reto_idx in range(N_RETOS):
    # Fijar el reto una sola vez
    resp_reto = sl_send({'accion': 'setear_reto', 'inport': INPORT, 'reto_idx': reto_idx})
    print(f'Reto {reto_idx+1:2d}/{N_RETOS}  ({resp_reto["status"]})')

    reto_data = {'reto_idx': reto_idx, 'sweeps': []}
    resultados['retos'].append(reto_data)   # referencia: los sweeps se van añadiendo en caliente

    for sw in range(N_SWEEPS):
        t1 = time.time()
        medidas = []
        for lam in lambdas:
            lam_r = round(float(lam), 4)
            laser.write(f'L={lam_r}')
            time.sleep(LASER_WAIT_S)
            resp = sl_send({'accion': 'medida_rapida'})
            medidas.append({'wavelength_nm': lam_r, 'powers': resp['powers']})

        reto_data['sweeps'].append({'sweep_idx': sw, 'medidas': medidas})

        # Guardar tras cada sweep completo (crash-safe)
        with open(fname, 'w') as f:
            json.dump(resultados, f, indent=2)

        print(f'    sweep {sw+1:3d}/{N_SWEEPS}  ({time.time()-t1:.0f}s)  guardado')

print(f'\nHecho. Total: {time.time()-t0:.0f}s')
print(f'Archivo: {fname}')

## 6. Visualización rápida

In [ ]:
lambdas_eje = [m['wavelength_nm'] for m in resultados['retos'][0]['sweeps'][0]['medidas']]

# Mostrar los N_SWEEPS barridos del reto 0 (estabilidad) y los sweep 0 de cada reto (huella)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Izquierda: todos los sweeps del reto 0
for sw_data in resultados['retos'][0]['sweeps']:
    medias = [np.mean(list(m['powers'].values())) for m in sw_data['medidas']]
    ax1.plot(lambdas_eje, medias, alpha=0.5)
ax1.set_xlabel('Longitud de onda (nm)')
ax1.set_ylabel('Potencia media (dBm)')
ax1.set_title(f'Estabilidad reto 0 ({N_SWEEPS} sweeps)')
ax1.grid(True, alpha=0.3)

# Derecha: sweep 0 de cada reto (huella espectral por reto)
for reto_data in resultados['retos']:
    medias = [np.mean(list(m['powers'].values())) for m in reto_data['sweeps'][0]['medidas']]
    ax2.plot(lambdas_eje, medias, alpha=0.6, label=f"reto {reto_data['reto_idx']}")
ax2.set_xlabel('Longitud de onda (nm)')
ax2.set_ylabel('Potencia media (dBm)')
ax2.set_title(f'Huella espectral por reto (sweep 0, {N_RETOS} retos)')
ax2.grid(True, alpha=0.3)

plt.suptitle(f'Barrido lambda, inport={INPORT}', y=1.02)
plt.tight_layout()
plt.show()

## 7. Cerrar instrumentos

In [ ]:
try:
    chan.sendall(b'EXIT\n')
    time.sleep(0.5)
except Exception:
    pass
ssh.close()
print('SSH cerrado.')

laser.write('DISABLE')
laser.close()
print('Laser apagado.')